# ?? D?tection de Fraudes par Carte Bancaire

1. Compr?hension du Probl?me

  * Objectif : d?tecter les transactions frauduleuses.

  * Jeu de donn?es tr?s d?s?quilibr? : 492 fraudes sur 284 807 transactions.

  * Les colonnes V1 ? V28 sont anonymis?es (issues d?une PCA = Analyse en Composantes Principales).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, precision_recall_curve)

from imblearn.over_sampling import SMOTE


In [ ]:
# Chargement des donn?es
df = pd.read_csv("creditcard.csv")
print("Dimensions du jeu de donn?es :", df.shape)

In [ ]:
# display the first 5 rows
print(df.head())

In [ ]:
# display info about the dataset
print(df.info())

In [ ]:
# display statistics of the dataset
df.describe().T

## Analyse Exploratoire (EDA)

In [ ]:
# R?partition des classes (fraude ou normale)
plt.figure(figsize=(6,4))
sns.countplot(x='Class', data=df)
plt.title('R?partition des classes : 0 = Normal, 1 = Fraude')
plt.show()

In [ ]:
# Montant des transactions par classe
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.boxplot(x='Class', y='Amount', data=df)
plt.title("Montant des transactions par classe")
plt.ylim(0, 300)  # Pour zoomer sur la majorit? des transactions

plt.subplot(1,2,2)
sns.histplot(df[df['Class']==1]['Amount'], bins=50)
plt.title("Montant des transactions frauduleuses")
plt.tight_layout()
plt.show()


## Pr?paration des Donn?es

In [ ]:
# Normalisation (scaling) de Time et Amount
scaler = RobustScaler()
df['montant_norm'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['temps_norm'] = scaler.fit_transform(df['Time'].values.reshape(-1, 1))

# Suppression des colonnes originales
df.drop(['Time', 'Amount'], axis=1, inplace=True)

# D?finition des variables explicatives et cible
X = df.drop('Class', axis=1)
y = df['Class']

# S?paration entra?nement / test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


## Mod?lisation

In [ ]:
# fonction d'evaluation de mod?le
def evaluer_modele(modele, X_test, y_test):
    from sklearn.metrics import fbeta_score

    y_pred = modele.predict(X_test)
    y_proba = modele.predict_proba(X_test)[:, 1]
    
    print("Pr?cision :", round(precision_score(y_test, y_pred), 3))
    print("Rappel :", round(recall_score(y_test, y_pred), 3))
    print("F2-score :", round(fbeta_score(y_test, y_pred, beta=2), 3))
    print("AUC ROC :", round(roc_auc_score(y_test, y_proba), 3))
    
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Matrice de confusion")
    plt.show()
    
    precision, rappel, _ = precision_recall_curve(y_test, y_proba)
    plt.plot(rappel, precision)
    plt.xlabel("Rappel")
    plt.ylabel("Pr?cision")
    plt.title("Courbe Pr?cision-Rappel")
    plt.show()


In [ ]:
xgb = XGBClassifier(
    scale_pos_weight=len(y_train[y_train == 0]) / len(y_train[y_train == 1]),
    eval_metric='aucpr',
    random_state=42
)
xgb.fit(X_train, y_train)
print("XGBoost :")
evaluer_modele(xgb, X_test, y_test)


## Ajustement du Seuil (XGBoost)

In [ ]:
# Probabilit?s pr?dites
y_proba = xgb.predict_proba(X_test)[:,1]

# Tester plusieurs seuils pour maximiser le F2-score
seuils = np.linspace(0.01, 0.5, 50)
f2_scores = []

from sklearn.metrics import fbeta_score

for s in seuils:
    y_pred_seuil = (y_proba >= s).astype(int)
    f2_scores.append(fbeta_score(y_test, y_pred_seuil, beta=2))

# Seuil optimal
seuil_optimal = seuils[np.argmax(f2_scores)]
print("Seuil optimal :", round(seuil_optimal, 3))

# ?valuer avec le seuil optimal
y_pred_opt = (y_proba >= seuil_optimal).astype(int)
print("F2-score :", round(fbeta_score(y_test, y_pred_opt, beta=2), 3))
print("Rappel :", round(recall_score(y_test, y_pred_opt), 3))
print("F1-score :", round(f1_score(y_test, y_pred_opt), 3))


# ?? Justification du choix du mod?le et discussion des r?sultats

## ?? Justification du choix du mod?le : XGBoost

Nous avons choisi **XGBoost** (Extreme Gradient Boosting) comme mod?le principal pour la d?tection de fraudes par carte bancaire en raison de plusieurs atouts :

1. **Performance ?lev?e sur les donn?es d?s?quilibr?es** :  
   La fraude est un ph?nom?ne rare. XGBoost int?gre des m?canismes comme la pond?ration des classes, la r?gularisation, et une capacit? ? capturer des relations non lin?aires complexes, ce qui le rend tr?s adapt?.

2. **Robustesse et stabilit?** :  
   Contrairement ? des mod?les plus simples comme la r?gression logistique, XGBoost est plus r?sistant aux valeurs aberrantes et peut mod?liser des interactions subtiles entre les variables.

3. **Excellents r?sultats obtenus** :  
   Les m?triques obtenues (voir ci-dessous) montrent que XGBoost d?tecte efficacement les fraudes tout en maintenant un bon compromis entre faux positifs et faux n?gatifs.

---

## ?? Analyse des r?sultats

### Seuil de classification optimal : `0.34`

Le mod?le pr?dit une transaction comme frauduleuse si la probabilit? d?passe **0.34** (au lieu de 0.5 par d?faut). Ce choix optimise le **F2-score**, ce qui est coh?rent dans un contexte o? **le rappel est prioritaire** (mieux vaut pr?venir une fraude que passer ? c?t?).

### R?sultats obtenus

| Indicateur       | Valeur | Interpr?tation |
|------------------|--------|----------------|
| **Pr?cision**     | 0.883  | Sur 100 alertes, environ **88** sont r?ellement des fraudes. |
| **Rappel**        | 0.847  | Le mod?le identifie **84.7 %** des fraudes r?elles. |
| **F1-score**      | 0.862  | Bon ?quilibre entre pr?cision et rappel. |
| **F2-score**      | 0.854  | Le mod?le donne plus d?importance au **rappel** qu?? la pr?cision. |
| **AUC ROC**       | 0.965  | Excellente capacit? de discrimination entre fraude et non-fraude. |

---

## ?? Implications pratiques

### ? Points positifs :
- Tr?s bonnes performances pour **r?duire les pertes financi?res**.
- Le seuil abaiss? ? `0.34` permet une **d?tection plus pr?coce** des fraudes.
- Un **F2-score ?lev?** montre que le mod?le privil?gie la **s?curit?**.

### ?? Limites et vigilance :
- **Faux positifs possibles** : certaines transactions normales peuvent ?tre signal?es ? tort.
- N?cessit? d'une **analyse m?tier** pour d?cider comment g?rer les alertes (blocage, alerte client, v?rification manuelle...).
- En production, il faudra un **syst?me de surveillance** pour ajuster dynamiquement le seuil.

---

## ?? Conclusion

Le choix de **XGBoost** s?appuie sur sa capacit? ? g?rer les d?s?quilibres, sa flexibilit?, et ses excellentes performances.  
Ce mod?le permet une **d?tection proactive des fraudes**, ? condition d?accompagner son usage de **proc?dures m?tier adapt?es**.

